In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.tsa.api import VAR
from sklearn.preprocessing import StandardScaler
import pickle

# leer datos

In [3]:
df_basa_full=pd.read_csv('basa_original.csv')

non_aquatic_species = 'Abies,Pinus,Juniperus,Taxus,Betula,Corylus,Alnus,Carpinus,Salix,Ulmus,Populus,Acer,Fraxinus,Fagus,Tilia,Juglans,Castanea,Quercus caducifolio,Quercus perennifolio,Pistacia,Rhamnus,Phillyrea,Buxus,Sambucus,Viburnum,Sanguisorba,Tamarix,Thymelaeaceae,Ephedra distachya,Ephedra fragilis,Ericaceae,Hereda helix,Ilex aquifolium,Viscum album,Lonicera,Vitis,Oleaceae,Myrtus,Olea,Poaceae,Lygeum spartum,Artemisia,Cichorioideae,Asteroideae,Cardueae,Rubiaceae,Centaurea,Chenopodiaceae,Caryophyllaceae,Plantago,Brassicaceae,Saxifragaceae,Fabaceae,Genista,Lotus type,Trifolium type,Rosaceae,Ribes,Boraginaceae,Sedum,Helianthemum,Lamiaceae,Urticaceae,Rumex,Berberidaceae,Euphorbiaceae,Primulaceae,Scrophulariaceae,Papaver,Campanulaceae,Convolvulaceae,Liliaceae,Iridaceae,Crassulaceae,Ranunculaceae,Cistaceae,Galium,Apiaceae,Valerianaceae,Cerealia type,Polygonaceae,Ranunculus'

species_mapping = {
        "Dec_Querc": "Quercus caducifolio",
        "Ever_Querc": "Quercus perennifolio",
        "Ephedra dist": "Ephedra distachya",
        "Ephedra frag": "Ephedra fragilis",
        "Lygeum": "Lygeum spartum",
        "Cicho": "Cichorioideae",
        "Astroi": "Asteroideae",
        "Carduaceae": "Cardueae",
        "Rubiac": "Rubiaceae",
        "Chenopo": "Chenopodiaceae",
        "Caryphy": "Caryophyllaceae",
        "Brassicac": "Brassicaceae",
        "Saxifrag": "Saxifragaceae",
        "Boraginac": "Boraginaceae",
        "Helianthem": "Helianthemum",
        "Euphorbiac": "Euphorbiaceae",
        "Primulac": "Primulaceae",
        "Scrophulari": "Scrophulariaceae",
        "Campanulac": "Campanulaceae",
        "Valerian": "Valerianaceae",
        "Cerealia": "Cerealia type",
        "Polygon": "Polygonaceae"
    }

non_aquatic_species = [s.strip() for s in non_aquatic_species.split(',')]
df_bas = df_basa_full[
    list(df_basa_full.columns[:8]) +
    [c for c in df_basa_full.columns[8:] if c in non_aquatic_species]
]

# sin variables condicionales

In [ ]:
N_BOOTSTRAP=500
MIN_NO_CERO=10
ALPHA=0.05
ABUNDANCIA_MINIMA=1

WINDOWS = {
    "9798_6253": (9798, 6253),
    "6182_3842": (6182, 3842),
    "3771_-57": (3771, -57)
}


def es_serie_valida(serie):
    return (serie > 0).sum() >= MIN_NO_CERO and serie.nunique() > 1


def bootstrap_univariante_exp(serie):
    n = len(serie)
    resultado = []

    while len(resultado) < n:
        inicio = np.random.randint(0, n)
        longitud = int(np.random.exponential(scale=L_MEDIA))
        longitud = max(1, min(longitud, n))
        indices = (inicio + np.arange(longitud)) % n
        resultado.extend(serie[indices])

    return np.array(resultado[:n])


def generar_bootstrap_df(df, columnas):
    df_boot = pd.DataFrame()
    df_boot["cal BP"] = df["cal BP"].values
    for col in columnas:
        df_boot[col] = bootstrap_univariante_exp(df[col].values)
    return df_boot


def extraer_ventana(df, cal_min, cal_max):
    mask = (df["cal BP"] >= min(cal_min, cal_max)) & \
           (df["cal BP"] <= max(cal_min, cal_max))
    return df[mask].copy().reset_index(drop=True)


def coef_sp1_a_sp2(df, sp1, sp2):
    try:
        x = df[sp1].values[::-1]
        y = df[sp2].values[::-1]

        if np.std(x) < MIN_VAR or np.std(y) < MIN_VAR:
            return np.nan

        data = pd.DataFrame({sp1: x, sp2: y})
        data_scaled = StandardScaler().fit_transform(data)

        model = VAR(data_scaled)
        results = model.fit(maxlags=1)
        A = results.coefs[0]

        return A[1][0]

    except:
        return np.nan


# cargar series temporales originales
with open("pkl/gam_original.pkl", "rb") as f:
    df_original = pickle.load(f)

especies= [col for col in df_original.columns if col != "cal BP"]
#transformar en 0 abundancias muy pequeñas
df_original[especies] = df_original[especies].applymap(lambda x: x if x >= ABUNDANCIA_MINIMA else 0)

# solo utilizar especies con más de MIN_NO_CERO abundancias mayores que 0 en la serie temporal original
especies_validas = [col for col in especies if es_serie_valida(df_original[col])]
print(f"Especies válidas: {especies_validas}")

# generar bootstrap
np.random.seed(42)
bootstrap_dfs = []
for i in range(N_BOOTSTRAP):
    df_boot = generar_bootstrap_df(df_original, especies_validas)
    bootstrap_dfs.append(df_boot)

# calcular coeficientes para cada ventana y cada par de especies 

resultados = {ventana: [] for ventana in WINDOWS}

for ventana, (cal_min, cal_max) in WINDOWS.items():
    for df_boot in bootstrap_dfs:
        df_ventana = extraer_ventana(df_boot, cal_min, cal_max)
        for i in range(len(especies_validas)):
            for j in range(i + 1, len(especies_validas)):
                sp1 = especies_validas[i]
                sp2 = especies_validas[j]
                coef = coef_sp1_a_sp2(df_ventana, sp1, sp2)
                resultados[ventana].append({
                    "sp1": sp1,
                    "sp2": sp2,
                    "coef": coef
                })
                






DUDAS

- Las abundancias hay que escalarlas? tiene sentido usarlas así  habría que hacer el porcentaje de cada especie respecto al total? o estandarizarlas? Cuánto tiene sentido que sea la mínima abundancia posible mayor que 0?



In [14]:
max(df_original[especies].values.flatten())

362.8415082765543